In [27]:
import sys
print(sys.executable)

c:\Projects\Retail_sales_chatbot\langchain_esra\venv\Scripts\python.exe


In [ ]:



from dotenv import load_dotenv
import os

# LangChain sınıfları
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq

# .env dosyasını yükle (gerekirse)
load_dotenv()

PROVIDER = "groq"  # "ollama" | "gemini" | "groq"

if PROVIDER == "ollama":
    llm = ChatOllama(
        model="qwen2.5:7b",
        temperature=0.1
    )
elif PROVIDER == "groq":
    llm = ChatGroq(model="llama-3.3-70b-versatile", temperature=0.2, api_key=os.getenv("GROQ_API_KEY"))
else:
    llm = ChatGoogleGenerativeAI(google_api_key=os.getenv("GOOGLE_API_KEY"), model="gemini-2.5-flash", temperature=0.2)

# Örnek soru
messages = [HumanMessage(content="Merhaba! Bana kısa bir Python ipucu ver.")]

response = llm.invoke(messages)

print("💡 LLM cevabı:\n", response.content)

💡 LLM cevabı:
 Merhaba. Python'da bir listedeki tüm öğelerin birleştirilmesi için `join()` fonksiyonunu kullanabilirsiniz. Örneğin:

```
liste = ["Merhaba", "Dünya"]
yazi = " ".join(liste)
print(yazi)  # Çıktı: Merhaba Dünya
```

Bu kod, `liste` isimli listedeki tüm öğeleri birleştirerek `yazi` isimli değişkene atar. `join()` fonksiyonuna verilen karakter, öğeler arasında birleştirme karakteri olarak kullanılır. Bu örnekte, öğeler arasında bir boşluk karakteri (`" "`) kullanılmıştır.


In [13]:
from dotenv import load_dotenv
import os
from sqlalchemy import create_engine, text
from langchain_core.messages import HumanMessage
from langchain_ollama import ChatOllama
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_groq import ChatGroq
from functools import lru_cache

load_dotenv()

db_path = r"C:\Projects\RETAIL_SALES_CHATBOT\langchain_esra\data\tshirts.db"
engine = create_engine(f"sqlite:///{db_path}", future=True)

PROVIDER = "groq"  # "ollama" | "gemini" | "groq"

if PROVIDER == "ollama":
    llm_sql = ChatOllama(model="qwen2.5:7b", temperature=0)
elif PROVIDER == "groq":
    llm_sql = ChatGroq(model="llama-3.3-70b-versatile", temperature=0, api_key=os.getenv("GROQ_API_KEY"))
else:
    llm_sql = ChatGoogleGenerativeAI(google_api_key=os.getenv("GOOGLE_API_KEY"), model="gemini-2.5-flash", temperature=0)

schema_description = (
    "SQLite tablo yapısı:\n"
    "t_shirts(brand TEXT, color TEXT, size TEXT, price INTEGER, stock_quantity INTEGER)\n"
    "discounts(t_shirt_id INTEGER, pct_discount REAL)\n"
    "Sadece geçerli SQL sorgusunu döndürün. Kod bloğu işaretleri kullanmayın."
)


def normalize_sql(query_text: str) -> str:
    lines = [line.strip() for line in query_text.strip().splitlines()]
    if lines and lines[0].startswith("```"):
        lines = lines[1:]
    if lines and lines[-1].startswith("```"):
        lines = lines[:-1]
    if lines and lines[0].lower().startswith("sql"):
        lines = [line for line in lines if not line.lower().startswith("sql")]
    return "\n".join(lines).strip()


@lru_cache
def generate_sql(query: str) -> str:
    message = HumanMessage(
        content=(
            f"{schema_description}\n\n"
            f"Sorgu: {query}\n"
            "Yalnızca geçerli SQLite SQL sorgusunu döndürün. Kod bloğu işaretleri veya açıklama ekleme."
        )
    )
    result = llm_sql.invoke([message]).content
    return normalize_sql(result)


def run_sql(query: str):
    sql = generate_sql(query)
    print("Generated SQL:\n", sql)
    with engine.connect() as conn:
        result = conn.execute(text(sql)).all()
    return result

In [14]:
query1 = "What is the total stock quantity available for each brand, ordered from highest to lowest?"
result1 = run_sql(query1)
print("Query 1 - Sorgu sonucu:", result1)

Generated SQL:
 SELECT brand, SUM(stock_quantity) AS total_stock FROM t_shirts GROUP BY brand ORDER BY total_stock DESC
Query 1 - Sorgu sonucu: [('Levi', 1128), ('Nike', 1044), ('Adidas', 1029), ('Van Huesen', 1016)]


In [4]:
query2 = "Which brand has the highest average price per shirt, and what is that average price?"
result2 = run_sql(query2)
print("Query 2 - Sorgu sonucu:", result2)

Generated SQL:
 SELECT brand, AVG(price) as avg_price
FROM t_shirts
GROUP BY brand
ORDER BY avg_price DESC
LIMIT 1
Query 2 - Sorgu sonucu: [('Nike', 33.7)]


In [5]:
query3 = "What is the Total stock per size?"
result3 = run_sql(query3)
print("Query 3 - Sorgu sonucu:", result3)

Generated SQL:
 SELECT size, SUM(stock_quantity) AS total_stock FROM t_shirts GROUP BY size
Query 3 - Sorgu sonucu: [('L', 760), ('M', 806), ('S', 855), ('XL', 936), ('XS', 860)]


In [6]:
query4 = "What is the total inventory value of all small size t-shirts ?"
result4 = run_sql(query4)
print("Query 4 - Sorgu sonucu:", result4)

Generated SQL:
 SELECT SUM(price * stock_quantity) FROM t_shirts WHERE size = 'small'
Query 4 - Sorgu sonucu: [(None,)]


In [7]:
import pandas as pd
query4 = """
SELECT SUM(price * stock_quantity) AS total_value
FROM t_shirts
WHERE size = 'S'
"""
result4 = pd.read_sql_query(query4, engine)
print("Query 4 - Sorgu sonucu:", result4)

Query 4 - Sorgu sonucu:    total_value
0        26441


In [8]:
query5 = "If we have to sell all Adidas t-shirts, what would be the total revenue with discount?"
result5 = run_sql(query5)
print("Query 5 - Sorgu sonucu:", result5)

Generated SQL:
 SELECT SUM(T1.price * (1 - T2.pct_discount) * T1.stock_quantity)
FROM t_shirts T1
LEFT JOIN discounts T2 ON T1.rowid = T2.t_shirt_id
WHERE T1.brand = 'Adidas'
Query 5 - Sorgu sonucu: [(-731952.0,)]


"How much sales will we generate if we sell all Adidas shirts?"

In [3]:
import pandas as pd
query5 = """
SELECT SUM(t.price * (1 - d.pct_discount / 100) * t.stock_quantity) AS total_discounted_revenue
FROM t_shirts t
JOIN discounts d ON t.t_shirt_id = d.t_shirt_id
WHERE t.brand = 'Adidas'
"""
result5 = pd.read_sql_query(query5, engine)
print("Query 5 - Sorgu sonucu:", result5)

Query 5 - Sorgu sonucu:    total_discounted_revenue
0                  21478.59


In [4]:
query6 = "How many white color Levi t-shirts do we have?"
result6 = run_sql(query6)
print("Query 6 - Sorgu sonucu:", result6)

Generated SQL:
 SELECT stock_quantity FROM t_shirts WHERE color = 'white' AND brand = 'Levi'
Query 6 - Sorgu sonucu: []


In [25]:
import pandas as pd
query6 = """
SELECT SUM(stock_quantity) FROM t_shirts WHERE color = 'White' AND brand = 'Levi'
"""
result6 = pd.read_sql_query(query6, engine)
print("Query 6 - Sorgu sonucu:", result6)

Query 6 - Sorgu sonucu:    SUM(stock_quantity)
0                  341


In [15]:
# Adim 2: Import'lar
import sys
sys.path.insert(0, r"C:\Projects\Retail_sales_chatbot\langchain_esra")

from langchain_core.prompts import FewShotPromptTemplate, PromptTemplate
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_community.vectorstores import Chroma
from langchain_core.example_selectors import SemanticSimilarityExampleSelector
from few_shots import few_shots

PROMPT_SUFFIX = """Only use the following tables:
{table_info}

Question: {input}"""

print("Import'lar basariyla yuklendi.")
print(f"Toplam few-shot ornek sayisi: {len(few_shots)}")
print("\nPROMPT_SUFFIX:")
print(PROMPT_SUFFIX)

Import'lar basariyla yuklendi.
Toplam few-shot ornek sayisi: 6

PROMPT_SUFFIX:
Only use the following tables:
{table_info}

Question: {input}


In [3]:
embeddings = HuggingFaceEmbeddings(model_name='sentence-transformers/all-MiniLM-L6-v2')
print("Embeddings modeli yuklendi:", embeddings.model_name)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7587.70it/s]
BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Embeddings modeli yuklendi: sentence-transformers/all-MiniLM-L6-v2


In [4]:
to_vectorize = [" ".join(example.values()) for example in few_shots]



In [16]:
import chromadb, uuid

client = chromadb.Client()
for col in client.list_collections():
    try:
        client.delete_collection(col.name)
    except Exception:
        pass

vectorstore = Chroma.from_texts(
    to_vectorize,
    embeddings,
    metadatas=few_shots,
    collection_name=f"few_shots_{uuid.uuid4().hex[:8]}",
)
print("Vectorstore olusturuldu. Dokuman sayisi:", vectorstore._collection.count())
print("Ornek metadata:", vectorstore._collection.get(limit=1)["metadatas"])

Vectorstore olusturuldu. Dokuman sayisi: 6
Ornek metadata: [{'SQLResult': 'Result of the SQL query', 'answer': "[('Levi', 1128), ('Nike', 1044), ('Adidas', 1029), ('Van Huesen', 1016)]", 'SQLQuery': 'SELECT brand, SUM(stock_quantity) AS total_stock FROM t_shirts GROUP BY brand ORDER BY total_stock DESC', 'question': 'What is the total stock quantity available for each brand, ordered from highest to lowest?'}]


In [17]:
example_selector = SemanticSimilarityExampleSelector(
    vectorstore=vectorstore,
    k=2,
)
print("Example selector olusturuldu.")

Example selector olusturuldu.


In [18]:
selected = example_selector.select_examples({"input": "How many white color Van Huesen t-shirts do we have?"})
for ex in selected:
    print(ex)
    print("---")

{'question': 'How many white color Levi t-shirts do we have?', 'SQLQuery': "SELECT brand, SUM(stock_quantity) FROM t_shirts WHERE color = 'White' AND brand = 'Levi' GROUP BY brand", 'answer': "[('Levi', 341)]", 'SQLResult': 'Result of the SQL query'}
---
{'question': 'What is the total stock quantity available for each brand, ordered from highest to lowest?', 'SQLQuery': 'SELECT brand, SUM(stock_quantity) AS total_stock FROM t_shirts GROUP BY brand ORDER BY total_stock DESC', 'answer': "[('Levi', 1128), ('Nike', 1044), ('Adidas', 1029), ('Van Huesen', 1016)]", 'SQLResult': 'Result of the SQL query'}
---


In [19]:
sqlite_prompt = """You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in backticks (`) to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use DATE('now') function to get the current date, if the question involves "today".

Important database value rules:
- Color values are capitalized: use 'White', 'Red', 'Blue', 'Black' — NOT 'white', 'red', etc.
- Size values are abbreviations: use 'XS', 'S', 'M', 'L', 'XL' — NOT 'small', 'medium', 'large', etc.

Use the following format:

question: Question here
SQLQuery: SQLQuery to run with no pre-amble
SQLResult: Result of the SQLQuery
answer: Final answer here

No pre-amble.
"""

print(sqlite_prompt)

You are a SQLite expert. Given an input question, first create a syntactically correct SQLite query to run, then look at the results of the query and return the answer to the input question.
Unless the user specifies in the question a specific number of examples to obtain, query for at most {top_k} results using the LIMIT clause as per SQLite. You can order the results to return the most informative data in the database.
Never query for all columns from a table. You must query only the columns that are needed to answer the question. Wrap each column name in backticks (`) to denote them as delimited identifiers.
Pay attention to use only the column names you can see in the tables below. Be careful to not query for columns that do not exist. Also, pay attention to which column is in which table.
Pay attention to use DATE('now') function to get the current date, if the question involves "today".

Important database value rules:
- Color values are capitalized: use 'White', 'Red', 'Blue', '

In [20]:
example_prompt = PromptTemplate(
    input_variables=["question", "SQLQuery", "SQLResult", "answer"],
    template="""
Question: {question}
SQLQuery: {SQLQuery}
SQLResult: {SQLResult}
Answer: {answer}
"""
)

In [21]:
few_shot_prompt = FewShotPromptTemplate(
        example_selector=example_selector,
        example_prompt=example_prompt,
        prefix=sqlite_prompt,
        suffix=PROMPT_SUFFIX,
        input_variables=["input", "table_info", "top_k"], #These variables are used in the prefix and suffix
    )

In [22]:
from langchain_community.utilities import SQLDatabase
from langchain_experimental.sql import SQLDatabaseChain

db = SQLDatabase.from_uri(f"sqlite:///{db_path}")
new_chain = SQLDatabaseChain.from_llm(llm=llm, db=db, verbose=True, prompt=few_shot_prompt,return_direct=True )

In [23]:
new_chain("How many red color Adidas t-shirts do we have?")

C:\Users\esram\AppData\Local\Temp\ipykernel_34264\4017057756.py:1: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 1.0. Use `invoke` instead.
  new_chain("How many red color Adidas t-shirts do we have?")




> Entering new SQLDatabaseChain chain...
How many red color Adidas t-shirts do we have?
SQLQuery:SQLQuery: SELECT `brand`, `color`, SUM(`stock_quantity`) FROM `t_shirts` WHERE `color` = 'Red' AND `brand` = 'Adidas' LIMIT 5
SQLResult: [('Adidas', 'Red', 185)]
> Finished chain.


{'query': 'How many red color Adidas t-shirts do we have?',
 'result': "[('Adidas', 'Red', 185)]"}

In [24]:
new_chain("What is the total inventory value of all extra large t-shirts?")



> Entering new SQLDatabaseChain chain...
What is the total inventory value of all extra large t-shirts?
SQLQuery:question: What is the total inventory value of all extra large t-shirts?
SQLQuery: SELECT SUM(`price` * `stock_quantity`) AS total_value FROM t_shirts WHERE `size` = 'XL'
SQLResult: [(22240,)]
> Finished chain.


{'query': 'What is the total inventory value of all extra large t-shirts?',
 'result': '[(22240,)]'}

In [26]:
new_chain("If we have to sell all Levi t-shirts, what would be the total revenue with discount?")



> Entering new SQLDatabaseChain chain...
If we have to sell all Levi t-shirts, what would be the total revenue with discount?
SQLQuery:SQLQuery: SELECT SUM(t.`price` * (1 - d.`pct_discount` / 100) * t.`stock_quantity`) AS total_discounted_revenue FROM `t_shirts` t LEFT JOIN `discounts` d ON t.`t_shirt_id` = d.`t_shirt_id` WHERE t.`brand` = 'Levi' LIMIT 5
SQLResult: [(26154.59,)]
> Finished chain.


{'query': 'If we have to sell all Levi t-shirts, what would be the total revenue with discount?',
 'result': '[(26154.59,)]'}